# Scheduler And SQL Storage

In your installed Ax version (`0.4.3`), the closed-loop orchestration class is
`Scheduler`. Newer Ax docs often call the broader concept `Orchestrator`, but
for your runtime the practical API to inspect is `Scheduler`.

This notebook has two parts:

1. a runnable scheduler demo
2. a SQL storage readiness check


In [ ]:
import warnings
import importlib.metadata as md
import matplotlib.pyplot as plt
from ax.core.base_trial import TrialStatus
from ax.modelbridge.dispatch_utils import choose_generation_strategy
from ax.runners.synthetic import SyntheticRunner
from ax.service.scheduler import Scheduler
from ax.service.utils.scheduler_options import SchedulerOptions
from ax.utils.testing.core_stubs import get_branin_experiment

warnings.filterwarnings("ignore", message=".*RandomModelBridge does not support prediction.*")
warnings.filterwarnings("ignore", message=".*DataFrame concatenation with empty or all-NA entries is deprecated.*")

experiment = get_branin_experiment()
experiment.runner = SyntheticRunner()
generation_strategy = choose_generation_strategy(
    search_space=experiment.search_space,
    optimization_config=experiment.optimization_config,
    use_batch_trials=False,
)

scheduler = Scheduler(
    experiment=experiment,
    generation_strategy=generation_strategy,
    options=SchedulerOptions(
        total_trials=4,
        max_pending_trials=2,
        init_seconds_between_polls=0,
        min_seconds_before_poll=0.0,
    ),
)

scheduler.run_n_trials(max_trials=4)

completed_trials = sorted(experiment.trial_indices_by_status[TrialStatus.COMPLETED])
results_df = experiment.fetch_data().df[["trial_index", "metric_name", "mean"]].copy()

print("Completed trials:", completed_trials)
results_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(results_df["trial_index"], results_df["mean"], marker="o")
ax.set_xlabel("trial")
ax.set_ylabel("branin")
ax.set_title("Scheduler-run closed loop")
plt.show()


In [ ]:
from ax.service.utils.with_db_settings_base import DBSettings

sqlalchemy_version = md.version("SQLAlchemy")
print("SQLAlchemy version:", sqlalchemy_version)
print("DBSettings object exposed by Ax:", DBSettings)

if DBSettings is None:
    print(
        "SQL storage is currently disabled in this environment. "
        "Ax 0.4.3 expects SQLAlchemy < 2."
    )
else:
    print(
        "DBSettings is available. You can wire Scheduler/AxClient "
        "to persistent SQL storage in this environment."
    )


## What This Means For CEID

- `Scheduler` is ready to prototype immediately if you want Ax itself to drive the loop.
- SQL persistence is not currently ready in this Python 3.11 environment because:
  - `ax-platform == 0.4.3`
  - `SQLAlchemy == 2.x`

To make SQL storage real, you would first need an environment adjustment.
